# Phase 1.1: 70/15/15 split, detector retrain, image classifier, fusion, TEST-set evaluation (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajaybahuguna17/qc-defect-detector/blob/master/phase1-vision/notebooks/train_colab_phase1_1.ipynb)

Goal: near-zero **image-level** error. Runs end to end on a fresh Colab GPU runtime:

1. clone, pinned install (`phase1-vision/requirements.txt`)
2. `prepare_data.py --split 70/15/15` (seed 42, 210/45/45 per class, pairwise overlap = 0) and
   `build_cls_dataset.py` (ImageFolder copy of the **same** split)
3. retrain the detector: YOLOv8n, imgsz 320, 100 epochs, seed 42
4. train the classifier: YOLOv8n-cls (ImageNet-pretrained), imgsz 224, 100 epochs, patience 20, seed 42
5. export both to TFLite FP32 + INT8 (calibrated on **train**); classifier output asserted `[1, 6]`
6. run `.pt`, FP32, INT8 of both models on every val and test image
7. classifier `.pt` vs FP32 vs INT8 top-1 on **all** test images
8. choose the detector per-class thresholds and the classifier threshold on **VAL only**
9. report on **TEST only**: detector mAP (.pt at 320, FP32, INT8), classifier top-1 + confusion
   matrix (.pt, FP32, INT8), fused image-level accuracy per class (FP32, INT8)
10. save models, labels, `model_io.json` (both models), results JSON, `fusion_config.json`, plots to Drive

The Phase 1 run folder on Drive (`neu_yolov8n_320_e100_s42`) is **never touched**: this run uses a
new name, refuses to overwrite an existing folder, and checks the old folder is byte-identical at the end.

**How to run:** `Runtime -> Change runtime type -> GPU (T4)`, then `Runtime -> Run all` (~45-60 min).
Nothing falls back silently: any missing file or failed step raises.

## 0. Configuration

In [ ]:
# ---- edit here --------------------------------------------------------------
REPO_URL     = "https://github.com/ajaybahuguna17/qc-defect-detector.git"
BRANCH       = "master"
SEED         = 42
SPLIT        = "70/15/15"
DET_EPOCHS, DET_PATIENCE, DET_IMGSZ, DET_BATCH = 100, 30, 320, 32
CLS_EPOCHS, CLS_PATIENCE, CLS_IMGSZ, CLS_BATCH = 100, 20, 224, 64
CALIB_FRACTION = 0.3       # INT8 calibration: 30 % of TRAIN, for both models
BASE_CONF, NMS_IOU = 0.05, 0.45   # detector decode for the sweep (higher thresholds applied afterwards)
USE_DRIVE    = True
DRIVE_ROOT   = "/content/drive/MyDrive/qc-defect-detector/phase1"
OLD_RUN_NAME = "neu_yolov8n_320_e100_s42"        # Phase 1 run: read-only, never written
RUN_NAME     = "p11_split701515_det320_cls224_s42"
ALLOW_CPU    = False
# ------------------------------------------------------------------------------
assert RUN_NAME != OLD_RUN_NAME, "RUN_NAME must differ from the Phase 1 run"
import os, sys
from pathlib import Path
REPO_DIR   = Path("/content/qc-defect-detector")
PHASE1     = REPO_DIR / "phase1-vision"
WORK       = Path("/content/work11")
EXPORT_DET = WORK / "export" / "detector"
EXPORT_CLS = WORK / "export" / "classifier"
RESULTS_DIR = WORK / "results"
for d in (WORK, EXPORT_DET, EXPORT_CLS, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
DET_RUN, CLS_RUN = f"{RUN_NAME}_det", f"{RUN_NAME}_cls"
print("run name:", RUN_NAME)

## 1. GPU check, Google Drive, protect the Phase 1 folder

In [ ]:
import subprocess, torch, hashlib
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not available")
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available() and not ALLOW_CPU:
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> GPU, then Run all again (or set ALLOW_CPU=True).")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
DEVICE = 0 if torch.cuda.is_available() else "cpu"

def tree_fingerprint(root):
    root = Path(root)
    if not root.exists():
        return None
    out = {}
    for p in sorted(root.rglob("*")):
        if p.is_file():
            out[str(p.relative_to(root))] = hashlib.sha256(p.read_bytes()).hexdigest()
    return out

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DEST = Path(DRIVE_ROOT) / RUN_NAME
    OLD_DRIVE = Path(DRIVE_ROOT) / OLD_RUN_NAME
    if DRIVE_DEST.exists():
        raise FileExistsError(f"{DRIVE_DEST} already exists. Change RUN_NAME (or delete that folder yourself).")
    OLD_FINGERPRINT = tree_fingerprint(OLD_DRIVE)
    print("Drive output:", DRIVE_DEST)
    print("Phase 1 folder:", OLD_DRIVE, "->", "missing" if OLD_FINGERPRINT is None else f"{len(OLD_FINGERPRINT)} files fingerprinted")

## 2. Clone the repo and install pinned dependencies
If pip asks to restart the runtime: `Runtime -> Restart session`, then `Run all` again.

In [ ]:
import shutil
def run(cmd, **kw):
    print("$", " ".join(map(str, cmd)))
    subprocess.run([str(c) for c in cmd], check=True, **kw)

url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
    if tok:
        url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if REPO_DIR.exists():
    run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH])
    run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"])
else:
    run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO_DIR])
GIT_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", GIT_COMMIT)
run(["pip", "install", "-q", "-r", PHASE1 / "requirements.txt"])

import importlib.metadata as md_, platform
VERSIONS = {}
for pkg in ["ultralytics", "litert-torch", "ai-edge-litert", "ai-edge-quantizer", "torch", "torchvision",
            "numpy", "opencv-python", "opencv-python-headless", "pillow", "matplotlib"]:
    try:
        VERSIONS[pkg] = md_.version(pkg)
    except md_.PackageNotFoundError:
        pass
VERSIONS.update(python=sys.version.split()[0], cuda=torch.version.cuda, platform=platform.platform())
for k, v in VERSIONS.items():
    print(f"{k:24s} {v}")
assert VERSIONS["ultralytics"] == "8.4.198", "unexpected ultralytics version; requirements.txt not applied"
sys.path.insert(0, str(PHASE1 / "scripts"))
sys.path.insert(0, str(PHASE1))

## 3. 70/15/15 split (`prepare_data.py`) and the classifier ImageFolder copy (`build_cls_dataset.py`)

In [ ]:
import json
run([sys.executable, PHASE1 / "scripts" / "prepare_data.py",
     "--raw-dir", PHASE1 / "data" / "raw", "--dataset-dir", PHASE1 / "data" / "dataset",
     "--seed", SEED, "--split", SPLIT])
CLS_DIR = PHASE1 / "data" / "cls"
run([sys.executable, PHASE1 / "scripts" / "build_cls_dataset.py",
     "--dataset-dir", PHASE1 / "data" / "dataset", "--out-dir", CLS_DIR])

from build_cls_dataset import class_from_filename
from tflite_utils import NEU_CLASSES
DATA_YAML = (PHASE1 / "data" / "dataset" / "data.yaml").resolve()
assert "test: images/test" in DATA_YAML.read_text(), "data.yaml has no test entry"
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
def images(split):
    return sorted(p for p in (PHASE1 / "data" / "dataset" / "images" / split).iterdir() if p.suffix.lower() in IMG_EXT)
SPLIT_IMGS = {sp: images(sp) for sp in ("train", "val", "test")}
want = {"train": 210, "val": 45, "test": 45}
for sp, imgs in SPLIT_IMGS.items():
    per = {c: 0 for c in NEU_CLASSES}
    for p in imgs:
        per[class_from_filename(p.name)] += 1
    assert all(v == want[sp] for v in per.values()), f"{sp}: {per}"
    cls_n = sum(1 for c in NEU_CLASSES for p in (CLS_DIR / sp / c).iterdir() if p.suffix.lower() in IMG_EXT)
    assert cls_n == len(imgs), f"classifier {sp} has {cls_n} images, detector {len(imgs)}"
stems = {sp: {p.stem for p in v} for sp, v in SPLIT_IMGS.items()}
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not stems[a] & stems[b], f"{a}/{b} overlap"
DATASET = {"split": SPLIT, "seed": SEED, **{f"{sp}_images": len(v) for sp, v in SPLIT_IMGS.items()},
           "per_class": want, "pairwise_overlap": 0}
print(DATASET)

## 4. Retrain the detector (YOLOv8n, imgsz 320, seed 42)

In [ ]:
import time
from ultralytics import YOLO
t0 = time.time()
det_model = YOLO("yolov8n.pt")      # COCO-pretrained starting point (transfer learning)
det_model.train(data=str(DATA_YAML), epochs=DET_EPOCHS, patience=DET_PATIENCE, imgsz=DET_IMGSZ, batch=DET_BATCH,
                seed=SEED, deterministic=True, workers=2, device=DEVICE,
                project=str(WORK / "runs"), name=DET_RUN, exist_ok=True, plots=True)
DET_TRAIN_MIN = (time.time() - t0) / 60
DET_TRAIN_DIR = Path(det_model.trainer.save_dir)
DET_PT = DET_TRAIN_DIR / "weights" / "best.pt"
if not DET_PT.is_file():
    raise FileNotFoundError(DET_PT)
DET_EPOCHS_DONE = int(getattr(det_model.trainer, "epoch", -1)) + 1
print(f"detector trained in {DET_TRAIN_MIN:.1f} min, {DET_EPOCHS_DONE} epochs -> {DET_PT}")

## 5. Train the image classifier (YOLOv8n-cls, ImageNet-pretrained, imgsz 224, patience 20, seed 42)
Ultralytics validates on `data/cls/val` during training; `test` is not touched until step 9.

In [ ]:
t0 = time.time()
cls_model = YOLO("yolov8n-cls.pt")  # ImageNet-pretrained starting point
cls_model.train(data=str(CLS_DIR), epochs=CLS_EPOCHS, patience=CLS_PATIENCE, imgsz=CLS_IMGSZ, batch=CLS_BATCH,
                seed=SEED, deterministic=True, workers=2, device=DEVICE,
                project=str(WORK / "runs"), name=CLS_RUN, exist_ok=True, plots=True)
CLS_TRAIN_MIN = (time.time() - t0) / 60
CLS_TRAIN_DIR = Path(cls_model.trainer.save_dir)
CLS_PT = CLS_TRAIN_DIR / "weights" / "best.pt"
if not CLS_PT.is_file():
    raise FileNotFoundError(CLS_PT)
CLS_EPOCHS_DONE = int(getattr(cls_model.trainer, "epoch", -1)) + 1
det_best, cls_best = YOLO(str(DET_PT)), YOLO(str(CLS_PT))
assert [det_best.names[i] for i in sorted(det_best.names)] == NEU_CLASSES, det_best.names
assert [cls_best.names[i] for i in sorted(cls_best.names)] == NEU_CLASSES, cls_best.names
print(f"classifier trained in {CLS_TRAIN_MIN:.1f} min, {CLS_EPOCHS_DONE} epochs -> {CLS_PT}")
print("classifier transforms stored in the checkpoint:", getattr(cls_best.model, "transforms", None))

## 6. Export both models to TFLite FP32 + INT8 (INT8 calibrated on train)

In [ ]:
run([sys.executable, PHASE1 / "scripts" / "export_tflite.py", "--weights", DET_PT, "--data", DATA_YAML,
     "--imgsz", DET_IMGSZ, "--out-dir", EXPORT_DET, "--precisions", "fp32,int8",
     "--calib-split", "train", "--calib-fraction", CALIB_FRACTION])
run([sys.executable, PHASE1 / "scripts" / "export_cls_tflite.py", "--weights", CLS_PT, "--data", CLS_DIR,
     "--imgsz", CLS_IMGSZ, "--out-dir", EXPORT_CLS, "--precisions", "fp32,int8", "--calib-fraction", CALIB_FRACTION])

DET_IO = json.loads((EXPORT_DET / "model_io.json").read_text())
CLS_IO = json.loads((EXPORT_CLS / "model_io.json").read_text())
DET_TFLITE = {p: EXPORT_DET / DET_IO["models"][p]["file"] for p in ("fp32", "int8")}
CLS_TFLITE = {p: EXPORT_CLS / CLS_IO["models"][p]["file"] for p in ("fp32", "int8")}
for p in ("fp32", "int8"):
    d, c = DET_IO["models"][p], CLS_IO["models"][p]
    assert d["output_shape"][1] == 10, d["output_shape"]
    assert c["output_shape"] == [1, 6], c["output_shape"]
    print(f"detector   {p}: {d['size_bytes']/1e6:.2f} MB | input {d['input_shape']} {d['input_layout']} {d['input_dtype']} "
          f"quant {d['input_quant']} | output {d['output_shape']} {d['output_dtype']}")
    print(f"classifier {p}: {c['size_bytes']/1e6:.2f} MB | input {c['input_shape']} {c['input_layout']} {c['input_dtype']} "
          f"quant {c['input_quant']} | output {c['output_shape']} {c['output_dtype']} softmax={c['output_is_softmax']}")
print("classifier normalisation:", json.dumps(CLS_IO["preprocessing"], indent=2))

## 7. Run every model on every val and test image
Detector decoded at conf 0.05 + class-aware NMS 0.45 (thresholds applied later). Classifier: 6 softmax outputs.

In [ ]:
import numpy as np
from PIL import Image
from tflite_utils import (load_interpreter, describe_io, run_tflite_on_image, describe_cls_io,
                          assert_classifier_model, run_cls_tflite, cls_preprocess)

det_interps = {p: load_interpreter(f) for p, f in DET_TFLITE.items()}
det_ios = {p: describe_io(it) for p, it in det_interps.items()}
cls_interps = {p: load_interpreter(f) for p, f in CLS_TFLITE.items()}
cls_ios = {p: describe_cls_io(it) for p, it in cls_interps.items()}
for io in cls_ios.values():
    assert_classifier_model(io)

def det_pt(path):
    r = det_best.predict(str(path), imgsz=DET_IMGSZ, conf=BASE_CONF, iou=NMS_IOU, verbose=False)[0]
    return [{"class_name": NEU_CLASSES[int(c)], "confidence": float(s), "bbox_xyxy": [float(v) for v in b]}
            for b, s, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy())]

def cls_pt(path):
    return [float(v) for v in cls_best.predict(str(path), imgsz=CLS_IMGSZ, verbose=False)[0].probs.data.cpu().numpy()]

# parity check of the reference classifier pre-processing (what Android must do) against the .pt pipeline
import copy
cls_torch = copy.deepcopy(cls_best.model).float().eval().to("cpu")   # private copy: predict() may move the original to GPU
def cls_pt_refpre(path):
    x = cls_preprocess(np.asarray(Image.open(path).convert("RGB")), CLS_IMGSZ).transpose(2, 0, 1)[None]
    with torch.no_grad():
        out = cls_torch(torch.from_numpy(np.ascontiguousarray(x)))
    out = out[0] if isinstance(out, (list, tuple)) else out
    return out[0].cpu().numpy()

RECORDS = {sp: {m: [] for m in ("pt", "fp32", "int8")} for sp in ("val", "test")}
PREPROC_MAXDIFF = 0.0
for sp in ("val", "test"):
    for img in SPLIT_IMGS[sp]:
        w, h = Image.open(img).size
        base = {"image": img.name, "true_class": class_from_filename(img.name), "width": w, "height": h}
        p_pt = cls_pt(img)
        PREPROC_MAXDIFF = max(PREPROC_MAXDIFF, float(np.abs(np.array(p_pt) - cls_pt_refpre(img)).max()))
        RECORDS[sp]["pt"].append({**base, "detections": det_pt(img), "probs": p_pt})
        for p in ("fp32", "int8"):
            dets, _ = run_tflite_on_image(det_interps[p], det_ios[p], img, NEU_CLASSES, BASE_CONF, NMS_IOU)
            probs = run_cls_tflite(cls_interps[p], cls_ios[p], img)
            RECORDS[sp][p].append({**base, "detections": dets, "probs": [float(v) for v in probs]})
    print(f"{sp}: {len(SPLIT_IMGS[sp])} images x (pt, fp32, int8) done")
print(f"classifier: max |prob(.pt pipeline) - prob(reference pre-processing)| = {PREPROC_MAXDIFF:.5f}")
if PREPROC_MAXDIFF > 0.02:
    raise AssertionError("tflite_utils.cls_preprocess does not match the .pt pre-processing; fix it before Android uses it")

## 8. Classifier `.pt` vs FP32 vs INT8 on ALL test images

In [ ]:
def top1(v):
    return int(np.argmax(v))
CLS_AGREEMENT = {}
for p in ("fp32", "int8"):
    ref, oth = RECORDS["test"]["pt"], RECORDS["test"][p]
    same = sum(top1(a["probs"]) == top1(b["probs"]) for a, b in zip(ref, oth))
    maxdiff = max(float(np.abs(np.array(a["probs"]) - np.array(b["probs"])).max()) for a, b in zip(ref, oth))
    CLS_AGREEMENT[p] = {"images": len(ref), "same_top1_as_pt": same, "rate": round(same / len(ref), 4),
                        "max_abs_prob_diff": round(maxdiff, 5),
                        "disagreements": [a["image"] for a, b in zip(ref, oth) if top1(a["probs"]) != top1(b["probs"])]}
    print(f"{p}: same top-1 as .pt on {same}/{len(ref)} test images ({same/len(ref):.1%}), max |dprob| {maxdiff:.4f}")

## 9. Choose thresholds on VAL only
Detector per-class thresholds first (maximise correct - wrong images per class), then the classifier threshold (maximise fused accuracy). Both summed over FP32 and INT8 so one config serves both. Ties -> centre of the widest plateau. Test is not looked at here.

In [ ]:
import fusion_eval, fusion
from fusion_eval import choose_det_thresholds, choose_cls_threshold, make_config, evaluate, classifier_only_accuracy
val_sets = [RECORDS["val"]["fp32"], RECORDS["val"]["int8"]]
DET_SEL = choose_det_thresholds(val_sets)
CLS_SEL = choose_cls_threshold(val_sets, DET_SEL["thresholds"])
FUSION_CFG = make_config(DET_SEL["thresholds"], CLS_SEL["threshold"])
print("detector thresholds (val):", DET_SEL["thresholds"])
for c, rows in DET_SEL["sweep"].items():
    best = max(r["score"] for r in rows)
    print(f"  {c:16s} best correct-wrong {best} (of {2*45} possible) at " +
          ", ".join(f"{r['threshold']:.2f}" for r in rows if r["score"] == best))
print("classifier threshold (val):", CLS_SEL["threshold"])
VAL_EVAL = {p: evaluate(RECORDS["val"][p], FUSION_CFG) for p in ("pt", "fp32", "int8")}
for p, e in VAL_EVAL.items():
    print(f"  val fused {p}: {e['correct']}/{e['images']} ({e['accuracy']:.1%}), review {e['review']}")
FUSION_CONFIG_OUT = {"selected_on": "val", "run_name": RUN_NAME, "git_commit": GIT_COMMIT,
                     "cls_threshold": FUSION_CFG["cls_threshold"],
                     "det_class_min_confidence": FUSION_CFG["det_class_min_confidence"],
                     "det_min_confidence": FUSION_CFG["det_min_confidence"],
                     "rule": "fusion_eval.py docstring (plateau centre, FP32+INT8 val summed)"}
(RESULTS_DIR / "fusion_config.json").write_text(json.dumps(FUSION_CONFIG_OUT, indent=2))

## 10. TEST report: detector mAP, classifier top-1 + confusion matrix, fused image-level accuracy

In [ ]:
def det_map(weights, **kw):
    task = {"task": "detect"} if str(weights).endswith(".tflite") else {}
    m = YOLO(str(weights), **task).val(data=str(DATA_YAML), imgsz=DET_IMGSZ, split="test", plots=False, verbose=False,
                                       project=str(WORK / "runs"), name=f"{RUN_NAME}_test_{Path(weights).stem}", exist_ok=True, **kw)
    per = {}
    for i, c in enumerate(m.box.ap_class_index):
        pr, rc, ap50, ap = m.box.class_result(i)
        per[NEU_CLASSES[int(c)]] = {"precision": float(pr), "recall": float(rc), "mAP50": float(ap50), "mAP50_95": float(ap)}
    return {"mAP50": float(m.box.map50), "mAP50_95": float(m.box.map), "precision": float(m.box.mp),
            "recall": float(m.box.mr), "per_class": per, "args": {k: str(v) for k, v in kw.items()}}

DET_TEST = {"pt_320": det_map(DET_PT, rect=False, batch=DET_BATCH),          # fixed 320 input, like the TFLite
            "fp32": det_map(DET_TFLITE["fp32"], batch=1), "int8": det_map(DET_TFLITE["int8"], batch=1)}
print(f"{'detector':8s} {'mAP50':>7s} {'mAP50-95':>9s}  (TEST, {len(SPLIT_IMGS['test'])} images, 320 px)")
for k, v in DET_TEST.items():
    print(f"{k:8s} {v['mAP50']:7.4f} {v['mAP50_95']:9.4f}")

# classifier: top-1 + confusion on test, and Ultralytics' own test top-1 as a cross-check
CLS_TEST = {p: classifier_only_accuracy(RECORDS["test"][p]) for p in ("pt", "fp32", "int8")}
mcls = cls_best.val(data=str(CLS_DIR), split="test", imgsz=CLS_IMGSZ, plots=True, verbose=False,
                    project=str(WORK / "runs"), name=f"{CLS_RUN}_test", exist_ok=True)
CLS_TEST["pt_ultralytics_val"] = {"top1": float(mcls.top1), "top5": float(mcls.top5)}
def cls_confusion(records):
    cm = np.zeros((6, 6), int)
    for r in records:
        cm[NEU_CLASSES.index(r["true_class"]), top1(r["probs"])] += 1
    return cm
CLS_CM = {p: cls_confusion(RECORDS["test"][p]) for p in ("pt", "fp32", "int8")}
for p in ("pt", "fp32", "int8"):
    a = CLS_TEST[p]
    print(f"classifier {p}: top-1 {a['correct']}/{a['images']} ({a['accuracy']:.1%}) | "
          + " ".join(f"{c} {v['correct']}/{v['images']}" for c, v in a["per_class"].items()))
print("Ultralytics .pt test top-1:", CLS_TEST["pt_ultralytics_val"])

# fused image-level accuracy on test with the VAL-chosen config
FUSED_TEST = {p: evaluate(RECORDS["test"][p], FUSION_CFG) for p in ("pt", "fp32", "int8")}
DET_ONLY_TEST = {p: evaluate(RECORDS["test"][p], FUSION_CFG, use_classifier=False) for p in ("fp32", "int8")}
for p in ("pt", "fp32", "int8"):
    e = FUSED_TEST[p]
    print(f"FUSED {p}: {e['correct']}/{e['images']} ({e['accuracy']:.1%}) | review {e['review']} | decided by {e['decided_by']}")
    print("   " + " ".join(f"{c} {v['correct']}/{v['images']}" for c, v in e["per_class"].items()))
    for err in e["errors"]:
        print(f"   ERROR {err['image']}: -> {err['final_class']} ({err['source']}) | {err['reason']}")
for p, e in DET_ONLY_TEST.items():
    print(f"detector only {p}: {e['correct']}/{e['images']} ({e['accuracy']:.1%})")

In [ ]:
import matplotlib.pyplot as plt
short = ["Cr", "In", "Pa", "PS", "RS", "Sc"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, p in zip(axes, ("pt", "fp32", "int8")):
    cm = CLS_CM[p]
    ax.imshow(cm, cmap="Blues")
    for i in range(6):
        for j in range(6):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(6), short); ax.set_yticks(range(6), short)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.set_title(f"classifier {p} (test): {CLS_TEST[p]['accuracy']:.1%}")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "cls_confusion_test.png", dpi=120); plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
xs = np.arange(6); wdt = 0.2
series = [("detector only FP32", DET_ONLY_TEST["fp32"]), ("classifier FP32", CLS_TEST["fp32"]),
          ("fused FP32", FUSED_TEST["fp32"]), ("fused INT8", FUSED_TEST["int8"])]
for k, (lab, d) in enumerate(series):
    ax.bar(xs + (k - 1.5) * wdt, [d["per_class"][c]["accuracy"] for c in NEU_CLASSES], wdt, label=lab)
ax.set_xticks(xs, NEU_CLASSES, rotation=15); ax.set_ylim(0.5, 1.01); ax.set_ylabel("image-level accuracy (test)")
ax.legend(loc="lower left"); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "image_accuracy_test.png", dpi=120); plt.show()

## 11. Save models, labels, model_io.json, results and plots to Drive

In [ ]:
from datetime import datetime, timezone
RESULTS = {
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "git_commit": GIT_COMMIT, "gpu": GPU_NAME, "versions": VERSIONS, "run_name": RUN_NAME,
    "config": {"seed": SEED, "split": SPLIT,
               "detector": {"base": "yolov8n.pt (COCO)", "imgsz": DET_IMGSZ, "epochs": DET_EPOCHS, "patience": DET_PATIENCE, "batch": DET_BATCH},
               "classifier": {"base": "yolov8n-cls.pt (ImageNet)", "imgsz": CLS_IMGSZ, "epochs": CLS_EPOCHS, "patience": CLS_PATIENCE, "batch": CLS_BATCH},
               "int8_calibration": {"split": "train", "fraction": CALIB_FRACTION},
               "sweep_decode": {"base_conf": BASE_CONF, "nms_iou": NMS_IOU}},
    "dataset": DATASET,
    "training": {"detector_minutes": round(DET_TRAIN_MIN, 1), "detector_epochs_completed": DET_EPOCHS_DONE,
                 "classifier_minutes": round(CLS_TRAIN_MIN, 1), "classifier_epochs_completed": CLS_EPOCHS_DONE},
    "model_io": {"detector": DET_IO, "classifier": CLS_IO},
    "classifier_preprocessing_parity_max_abs_prob_diff": PREPROC_MAXDIFF,
    "threshold_selection_val": {"detector": DET_SEL, "classifier": CLS_SEL, "fusion_config": FUSION_CONFIG_OUT,
                                "val_fused": {p: {k: v for k, v in e.items()} for p, e in VAL_EVAL.items()}},
    "test": {"detector_map": DET_TEST, "classifier_top1": CLS_TEST,
             "classifier_confusion": {p: cm.tolist() for p, cm in CLS_CM.items()},
             "classifier_agreement_vs_pt": CLS_AGREEMENT,
             "fused_image_level": FUSED_TEST, "detector_only_image_level": DET_ONLY_TEST,
             "note": "NEU-DET has no defect-free images: false alarms on good parts are NOT measured."},
}
(RESULTS_DIR / "phase1_1_results.json").write_text(json.dumps(RESULTS, indent=2))

for src in [*DET_TRAIN_DIR.glob("*.png"), DET_TRAIN_DIR / "results.csv"]:
    if Path(src).is_file():
        shutil.copy2(src, RESULTS_DIR / f"det_{Path(src).name}")
for src in [*CLS_TRAIN_DIR.glob("*.png"), CLS_TRAIN_DIR / "results.csv"]:
    if Path(src).is_file():
        shutil.copy2(src, RESULTS_DIR / f"cls_{Path(src).name}")

MODELS_OUT = WORK / "models"
for sub, pt, exp in (("detector", DET_PT, EXPORT_DET), ("classifier", CLS_PT, EXPORT_CLS)):
    d = MODELS_OUT / sub
    d.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pt, d / f"best_{RUN_NAME}_{sub}.pt")
    for f in [*exp.glob("*.tflite"), exp / "labels.txt", exp / "model_io.json"]:
        shutil.copy2(f, d / f.name)

if USE_DRIVE:
    assert not DRIVE_DEST.exists(), DRIVE_DEST
    shutil.copytree(RESULTS_DIR, DRIVE_DEST / "results")
    shutil.copytree(MODELS_OUT, DRIVE_DEST / "models")
    assert tree_fingerprint(OLD_DRIVE) == OLD_FINGERPRINT, "the Phase 1 Drive folder changed!"
    print("Saved to Drive:", DRIVE_DEST, "| Phase 1 folder unchanged")
for p in sorted(MODELS_OUT.rglob("*")) + sorted(RESULTS_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.relative_to(WORK)}  ({p.stat().st_size/1e6:.2f} MB)")

## 12. After the run
1. Copy `<Drive>/p11_split701515_det320_cls224_s42/results/` into `phase1-vision/results/phase1_1/` and commit
   it (JSON + PNGs only; never `.pt` / `.tflite`).
2. Put the chosen thresholds from `fusion_config.json` into `FUSION_CONFIG` in `phase1-vision/fusion.py`.
3. Fill the classifier TODOs in `phase1-vision/MODEL_IO.md` from `models/classifier/model_io.json`
   and update the Phase 1.1 tracker in `PROJECT_CONTEXT.md` with the TEST numbers.